# Populate csv file with txt transcripts
## **need**
- csv with transcripts column that needs to be populated
- transcript column name
- txt title id structure (how to correlate txt title to the column name)
- folder location of txt files

## Inputs
- **csv_path**: path to the CSV file to update
- **transcript_col**: column name to populate (e.g., `transcript`)
- **txt_folder_path**: folder containing `.txt` files
- **id rule**: how the text filename maps to the CSV id (example below uses filename stem)

## Expected CSV columns
- An identifier column to match files (example: `id`)
- The transcript column to fill

## Notes
- This example reads each `.txt` file and joins on `id` == filename stem.
- Adjust `id_col` and `file_id_from_name()` to match your naming scheme.

In [10]:
import ast
import re
from pathlib import Path
import pandas as pd
import csv

def file_ids_from_name(path: Path) -> list[str]:
    stem = path.stem.strip()
    ids = [stem]

    parts = stem.split("-")
    if len(parts) >= 2:
        ids.insert(0, parts[-2].strip())

    return list(dict.fromkeys([i for i in ids if i]))

def read_transcript(path: Path) -> str:
    raw = path.read_text(encoding="utf-8-sig", errors="ignore")
    if not raw:
        return ""

    # Unwrap list-literal files if needed.
    raw = raw.strip()
    if raw.startswith("[") and raw.endswith("]"):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                raw = " ".join(str(item) for item in parsed)
        except (ValueError, SyntaxError):
            pass

    # Remove page markers.
    raw = re.sub(r"(?m)^===\s*Page\s*\d+\s*===\s*$", " ", raw)

    # Normalize all line endings.
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")

    # Flatten every line into one paragraph.
    lines = [line.strip() for line in raw.split("\n")]
    lines = [line for line in lines if line]

    text = " ".join(lines)

    # Fix OCR hyphenation across line breaks if it survived.
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)

    # Collapse extra whitespace.
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
csv_file = pd.read_csv(csv_path)

txt_dir = Path(txt_folder_path)
txt_map = {}

for p in txt_dir.glob("*.txt"):
    txt = read_transcript(p)
    for file_id in file_ids_from_name(p):
        txt_map.setdefault(file_id, txt)

id_series = csv_file[id_col].astype(str).str.strip()
mapped = id_series.map(txt_map)

fallback_ids = id_series.str.replace(r"\.txt$", "", regex=True)
mapped = mapped.fillna(fallback_ids.map(txt_map))

csv_file[transcript_col] = mapped.fillna("")

output_path = Path(csv_path).with_name(f"{name} metadata.csv")
csv_file.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)